<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>

# 12 · Consumo de APIs REST con Python

## Observatorio de auditoría · Contraste de conversiones USD→EUR

**Objetivo.** Consumir una API HTTP, interpretar sus respuestas, controlar fallos y convertir datos externos en evidencia que otra persona pueda recalcular.

**Cómo trabajar.** Abre el notebook desde la carpeta extraída del bloque y ejecuta las celdas en orden con un kernel Python 3.10 o superior. Los ejemplos usan solo biblioteca estándar y levantan una API en 127.0.0.1 con un puerto temporal. Los datos de operaciones y las cotizaciones de aula son ficticios. Los diez ejercicios aparecen al final.

**Mapa de lectura.** Primero trabajaremos con JSON, autenticación y paginación en local. Después interpretaremos el formato CSV del BCE y el criterio financiero del Observatorio. Los ejemplos de resultados que siguen corresponden al modo de aula.

**Fuente de contraste.** [API estadística del BCE](https://data.ecb.europa.eu/help/api/data), flujo EXR, serie D.USD.EUR.SP00.A. Es una referencia informativa. En una revisión real, el tipo aplicable puede depender del contrato, la hora, los costes y las reglas de la operación.

**Versión alumnado.** Los ejemplos están resueltos y los ejercicios contienen espacios para tu código.

In [1]:
from collections import Counter
from decimal import Decimal, ROUND_HALF_UP
from hashlib import sha256
from pathlib import Path
from subprocess import run
from tempfile import TemporaryDirectory
from urllib.error import HTTPError
from urllib.parse import parse_qs, urlencode, urlsplit
import json
import sys

BASE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

from api_auditoria import (
    DATOS, conciliar, get, leer_cotizaciones, leer_json,
    leer_operaciones, paginar, url_bce,
)
from servidor_aula import api_local


def estado_de_error(url, **opciones):
    # Conservar el código HTTP sin volcar credenciales ni el cuerpo de error.
    try:
        get(url, **opciones)
    except HTTPError as error:
        return error.code
    raise AssertionError("Se esperaba un error HTTP")

## Ejemplo 1 · Petición, respuesta y representación

El método GET pide una representación. La ruta identifica la colección. El servidor devuelve el estado, cabeceras y bytes. La lectura de JSON exige un tipo de contenido esperado. Aquí el campo `items` contiene dos filas y `next_page` indica que aún queda información.

**Resultado esperado.** Estado 200 y los identificadores OP-001 y OP-002. La siguiente página es 2. Una respuesta exitosa puede representar una sola página.

In [2]:
with api_local() as base:
    cuerpo, meta = get(f"{base}/operaciones")
    lote = leer_json(cuerpo, meta)
    print(meta["estado_http"], meta["content_type"])
    print([fila["id"] for fila in lote["items"]], lote["next_page"])

200 application/json; charset=utf-8
['OP-001', 'OP-002'] 2


## Ejemplo 2 · Construcción de la consulta

El origen identifica el servidor, la ruta selecciona el recurso y la consulta transporta valores documentados por la API. `urlencode` protege separadores al convertir valores en texto de URL. El servidor de aula admite `page` y `limit`.

**Resultado esperado.** La segunda página contiene OP-003 y OP-004. `next_page` es None porque JSON null se traduce a None.

In [3]:
consulta = urlencode({"page": 2, "limit": 2})
with api_local() as base:
    cuerpo, meta = get(f"{base}/operaciones?{consulta}")
    lote = leer_json(cuerpo, meta)
    print([fila["id"] for fila in lote["items"]], lote["next_page"])

['OP-003', 'OP-004'] None


## Ejemplo 3 · Errores de cliente

El servidor devuelve 400 cuando la paginación solicitada es inválida y 404 cuando la ruta falta. Python entrega ambos como HTTPError. Una excepción de transporte, como un fallo DNS, puede ocurrir antes de recibir un estado HTTP.

**Resultado esperado.** 400 seguido de 404. La pausa queda vacía porque la política de aula no reintenta 400.

In [4]:
pausas = []
with api_local() as base:
    print(estado_de_error(f"{base}/operaciones?page=0", esperar=pausas.append))
    print(estado_de_error(f"{base}/ruta-inexistente"))
print("Pausas:", pausas)

400
404
Pausas: []


## Ejemplo 4 · Autenticación de laboratorio

Bearer describe cómo presentar un token. El servidor de aula comprueba un valor ficticio. Un 401 requiere revisar la autenticación. Un 403 invita a comprobar la autorización concedida. En una integración, usa el mecanismo de custodia autorizado y evita imprimir el token.

**Resultado esperado.** 401 sin token y un JSON de acceso autorizado con el token de aula.

In [4]:
with api_local() as base:
    print(estado_de_error(f"{base}/privado"))
    cuerpo, meta = get(f"{base}/privado", token="token-aula")
    print(leer_json(cuerpo, meta))

401
{'acceso': 'autorizado'}


## Ejemplo 5 · 429, espera y presupuesto

El endpoint de prueba devuelve 429 y `Retry-After: 2` en el primer intento de cada servidor nuevo. El cliente elige la mayor pausa entre el backoff y la cabecera. Inyectar una función para registrar las pausas permite comprobar la política sin esperar de verdad.

**Resultado esperado.** Dos intentos, una pausa planificada de dos segundos y respuesta final recuperada. Con un único intento, el 429 llega al llamador.

In [5]:
pausas = []
with api_local() as base:
    cuerpo, meta = get(f"{base}/limite", esperar=pausas.append)
    print(meta["intentos"], pausas, leer_json(cuerpo, meta))
with api_local() as base:
    print(estado_de_error(f"{base}/limite", intentos=1))

2 [2] {'estado': 'recuperado'}
429


## Ejemplo 6 · Paginación y reconciliación de cobertura

La colección completa requiere seguir el marcador de continuación hasta null. El cliente protege contra cursores repetidos, IDs duplicados y exceso de páginas. La deduplicación silenciosa podría ocultar un fallo de extracción.

**Resultado esperado.** Los cuatro IDs en orden. Con límite de una página, se informa del fallo y se descarta el resultado parcial.

In [7]:
with api_local() as base:
    filas = paginar(base, limite=2)
    print([fila["id"] for fila in filas])
    try:
        paginar(base, limite=2, max_paginas=1)
    except ValueError as error:
        print(error)

['OP-001', 'OP-002', 'OP-003', 'OP-004']
Se alcanzó el límite de páginas


## Ejemplo 7 · Contrato del cuerpo JSON

Hay tres controles sucesivos: tipo de contenido, sintaxis JSON y estructura de negocio. Un diccionario JSON válido puede carecer de `items`. El cliente genérico lee JSON, mientras `paginar` valida el contrato de la colección.

**Resultado esperado.** El tipo text/html se rechaza. El JSON sintáctico que omite items se puede interpretar, pero el consumidor de la colección debe rechazarlo.

In [6]:
try:
    leer_json(b"<html>Error del proxy</html>", {"content_type": "text/html"})
except ValueError as error:
    print(error)
documento = leer_json(b'{"mensaje":"ok"}', {"content_type": "application/json"})
print("Tiene items:", "items" in documento)

Respuesta inesperada: text/html
Tiene items: False


## Ejemplo 8 · SDMX, fechas y unidades

EXR es el flujo. D identifica la frecuencia diaria, USD la moneda observada y EUR la denominadora. SP00 corresponde al tipo spot y A es el sufijo de la serie. `TIME_PERIOD` identifica la fecha y `OBS_VALUE` expresa USD por 1 EUR. El cliente valida dimensiones, fechas, valores positivos finitos y ausencia de duplicados.

**Resultado esperado.** Se construye la consulta pública de la ventana de aula. Se leen tres observaciones sintéticas. El 6 de enero carece de observación en el fichero local.

In [9]:
consulta_bce = url_bce("2024-01-02", "2024-01-06")
print(urlsplit(consulta_bce).path)
print(parse_qs(urlsplit(consulta_bce).query))
csv_aula = (DATOS / "bce_simulado.csv").read_bytes()
tipos = leer_cotizaciones(csv_aula)
print(tipos)
print("Referencia para el 6:", tipos.get("2024-01-06"))

/service/data/EXR/D.USD.EUR.SP00.A
{'startPeriod': ['2024-01-02'], 'endPeriod': ['2024-01-06'], 'format': ['csvdata']}
{'2024-01-02': Decimal('1.1000'), '2024-01-03': Decimal('1.0800'), '2024-01-04': Decimal('1.0900')}
Referencia para el 6: None


## Ejemplo 9 · Cálculo monetario y umbral

Si 1 EUR vale 1,08 USD, dividir 1080 USD entre 1,08 USD/EUR produce 1000 EUR. Redondeamos el esperado al céntimo con ROUND_HALF_UP. La diferencia se calcula como registrado menos esperado. Los valores entran en Decimal directamente desde texto para preservar decimales de origen.

**Resultado esperado.** +15,00 EUR para OP-002. +0,01 EUR cae dentro del umbral inclusivo. El ejemplo de empate 1,005 se redondea a 1,01 con la política del aula.

In [14]:
esperado = (Decimal("1080.00") / Decimal("1.08")).quantize(
    Decimal("0.01"), rounding=ROUND_HALF_UP
)
delta = Decimal("1015.00") - esperado
print(esperado, delta, abs(delta) <= Decimal("0.01"))
print(Decimal("1.005").quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

1000.00 15.00 False
1.01


## Ejemplo 10 · Conciliación y lectura de hallazgos

El cruce por fecha exacta deja esperado y diferencia en None si falta una observación. En un informe, esta fila debe seguir visible. La cobertura del contraste usa el total de operaciones como denominador; la tasa de diferencias entre evaluables usa solo las que tienen referencia.

**Resultado esperado.** Dos dentro de tolerancia, una para revisar y una sin referencia. Cobertura 75 %. Tasa de revisión entre evaluables 33,3 %.

In [11]:
resultado = conciliar(leer_operaciones(), tipos)
for fila in resultado:
    print(fila["id"], fila["estado"], fila["diferencia_eur"])
conteo = Counter(fila["estado"] for fila in resultado)
evaluables = len(resultado) - conteo["sin_referencia"]
print(f"Cobertura: {evaluables / len(resultado):.1%}")
print(f"Revisar entre evaluables: {conteo['revisar'] / evaluables:.1%}")

OP-001 dentro_tolerancia 0.00
OP-002 revisar 15.00
OP-003 dentro_tolerancia 0.01
OP-004 sin_referencia None
Cobertura: 75.0%
Revisar entre evaluables: 33.3%


## Ejemplo 11 · Evidencia y ejecución integrada

El comando conserva los bytes de entrada y guarda su SHA-256 en el JSON del control. La comparación detecta cambios respecto a la huella registrada. Para reproducibilidad plena en un proceso real, conserva además los datos internos, parámetros, versión del código y permisos de custodia.

**Resultado esperado.** Dos archivos y coincidencia de la huella. El ejemplo usa una carpeta temporal que se elimina al terminar. Para conservar archivos, ejecuta el comando con un directorio nuevo de tu elección.

In [12]:
with TemporaryDirectory() as temporal:
    destino = Path(temporal) / "evidencias"
    run([sys.executable, str(BASE / "api_auditoria.py"), str(destino)], check=True)
    informe = json.loads((destino / "control.json").read_text(encoding="utf-8"))
    huella = sha256((destino / "respuesta.csv").read_bytes()).hexdigest()
    print(sorted(ruta.name for ruta in destino.iterdir()))
    print("Huella coincide:", huella == informe["fuente"]["sha256_csv"])
    print(informe["fuente"]["origen"])

['control.json', 'respuesta.csv']
Huella coincide: True
Cotizaciones sintéticas de aula


## Demostración opcional con el BCE

Desde la carpeta del bloque, ejecuta `python api_auditoria.py evidencias_bce --en-vivo`. Inspecciona la URL, el tipo de contenido y el recuento de resultados. Compara con `python api_auditoria.py evidencias_aula`. El proveedor puede revisar sus cifras, de modo que el criterio de aceptación del laboratorio se aplica al CSV sintético.

## Diez ejercicios

Trabaja en orden. En cada ejercicio comprueba el criterio de aceptación y redacta una frase que interprete el resultado en el contexto de auditoría.

### Ejercicio 01 · Desmontar la URL SDMX

Genera una consulta entre 2024-01-02 y 2024-01-06. Extrae y verifica el host, la clave de serie y los tres parámetros. Explica qué cambia si editas solo endPeriod.

**Pista.** Combina url_bce, urlsplit y parse_qs. Los valores de parse_qs son listas.

**Criterio de aceptación.** El host es data-api.ecb.europa.eu y la consulta contiene startPeriod, endPeriod y format=csvdata. La fecha final acota la ventana de la misma serie.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 02 · Recuperar una página concreta

Consulta page=2 y limit=2, verifica el 200, extrae los identificadores y decide si queda otra página.

**Pista.** Codifica los parámetros con urlencode y usa leer_json.

**Criterio de aceptación.** OP-003, OP-004 y next_page=None. La respuesta cierra la paginación.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 03 · Distinguir 400 y 404

Solicita limit=0 en una ruta válida y después una ruta inexistente. Obtén ambos estados y comprueba que no se programan pausas.

**Pista.** Pasa esperar=pausas.append a estado_de_error en ambas llamadas.

**Criterio de aceptación.** 400 exige corregir el parámetro. 404 exige revisar la ruta. Ninguno se reintenta.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 04 · Controlar autenticación

Prueba /privado con un token incorrecto y con el token de aula. Verifica ambos resultados. Registra los estados sin imprimir la cabecera Authorization.

**Pista.** Usa el argumento token. El cliente compone la cabecera Bearer.

**Criterio de aceptación.** 401 con token incorrecto y 200 con token-aula. El registro muestra únicamente los estados.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 05 · Respetar Retry-After

Ejecuta /limite con un máximo de dos intentos y captura la pausa sin dormir. Comprueba que un servidor nuevo con un solo intento devuelve 429.

**Pista.** Usa un contexto api_local nuevo para cada prueba. El servidor conserva el contador durante ese contexto.

**Criterio de aceptación.** Dos intentos, pausa [2] y éxito. Con un intento se propaga 429.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 06 · Medir el límite de paginación

Recupera toda la colección con tres registros por página. Repite con max_paginas=1 y verifica que el cliente señala el resultado incompleto.

**Pista.** Una página tendrá tres filas y otra una. Captura ValueError en la segunda prueba.

**Criterio de aceptación.** Cuatro IDs distintos en la extracción completa. El límite de una página lanza ValueError.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 07 · Detectar un cambio semántico

Cambia USD por GBP en una copia en memoria del CSV. Comprueba que el lector rechaza la serie. Explica por qué un CSV bien formado puede ser una entrada incorrecta.

**Pista.** Usa bytes.replace en la copia y deja el fichero del aula intacto.

**Criterio de aceptación.** El lector rechaza la serie modificada. Una moneda distinta invalidaría las unidades del cálculo.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 08 · Probar ambos lados de la tolerancia

Calcula OP-002 y OP-003. Comprueba sus diferencias y estados con 0,01 EUR. Repite OP-003 con tolerancia cero y explica el cambio.

**Pista.** Construye cotizaciones desde el CSV. Pasa Decimal('0.00') al argumento tolerancia para el segundo cálculo.

**Criterio de aceptación.** +15,00 requiere revisión. +0,01 está dentro con umbral inclusivo y requiere revisión si el umbral es cero.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 09 · Mantener visible la falta de referencia

Contrasta OP-004 con el CSV local. Añade un tipo sintético del 7 de enero y demuestra que el cruce por fecha exacta sigue sin referencia.

**Pista.** Copia el diccionario de tipos antes de añadir la fecha futura.

**Criterio de aceptación.** Ambas ejecuciones tienen estado sin_referencia y diferencia None. El tipo posterior no participa.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación:

### Ejercicio 10 · Entregar un control verificable

Genera evidencias en un directorio temporal. Verifica la huella, simula una alteración en memoria y resume los cuatro estados. Indica el denominador de la cobertura.

**Pista.** Usa run con check=True, sha256, json y Counter. Añade un byte al CSV en memoria.

**Criterio de aceptación.** Huella original válida, huella alterada distinta y conteo 2/1/1. La cobertura es tres de cuatro.

In [ ]:
# TODO: escribe y ejecuta tu solución.
# Interpretación: